# Pronóstico de ventas de supermercado: Store Sales (Favorita, Ecuador)

**Contexto:** proyecto de series de tiempo con los datos de la competencia de Kaggle [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting). El objetivo es predecir las ventas diarias de cada familia de producto en cada tienda de Favorita durante los **16 días** siguientes al último día de entrenamiento.

**Datos:** no se incluyen en este repositorio. Los descargo de Kaggle (hay que aceptar las reglas de la competencia) y los guardo en la carpeta `data/`. Según las reglas, el uso de los datos es solo para la competencia y para investigación y educación sin fines comerciales, y no se pueden redistribuir.

**Métrica:** RMSLE (error cuadrático medio logarítmico). Como el modelo trabaja sobre `log1p(ventas)`, equivale al RMSE en esa escala.

**Qué hago en este notebook**
1. Cargo los datos y reviso su estructura
2. Exploro tendencia, estacionalidad, familias, promociones y feriados
3. Diseño una validación que imita el problema real (últimos 16 días como prueba)
4. Construyo dos modelos de referencia y un modelo de gradient boosting con variables de rezago
5. Comparo errores y reviso dónde falla el modelo
6. Genero las predicciones para el conjunto de prueba de Kaggle

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# Silencio solo los FutureWarning de pandas/seaborn sobre APIs que cambiarán; los demás avisos se ven
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", "{:,.3f}".format)

DATA_DIR = Path("data")
IMG_DIR = Path("imagenes")
IMG_DIR.mkdir(exist_ok=True)

HORIZONTE = 16                          # días a predecir: el test va del 16 al 31 de agosto de 2017
LAG_MIN = HORIZONTE                     # antigüedad mínima de rezagos y medias móviles
VENTANA_CEROS = 28                      # días sin ventas para aplicar la regla de ceros
FECHA_INICIO_ENTRENAMIENTO = "2015-01-01"
MAX_ITER = 300                          # iteraciones del gradient boosting
LEARNING_RATE = 0.08
MAX_LEAF_NODES = 63
MIN_SAMPLES_LEAF = 100
L2_REG = 1.0
GENERAR_SUBMISSION = True               # genera submission.csv para subir a Kaggle
SEMILLA = 42

## 1. Carga y vista general

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])
stores = pd.read_csv(DATA_DIR / "stores.csv")
oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(DATA_DIR / "holidays_events.csv", parse_dates=["date"])

fin_train = train["date"].max()
resumen = pd.Series({
    "filas de train": f"{len(train):,}",
    "periodo de train": f"{train['date'].min():%Y-%m-%d} a {fin_train:%Y-%m-%d}",
    "periodo de test": f"{test['date'].min():%Y-%m-%d} a {test['date'].max():%Y-%m-%d}",
    "tiendas": train["store_nbr"].nunique(),
    "familias de producto": train["family"].nunique(),
    "series (tienda x familia)": train.groupby(["store_nbr", "family"]).ngroups,
}, name="valor")
display(resumen.to_frame())

calendario = pd.date_range(train["date"].min(), fin_train, freq="D")
dias_faltantes = calendario.difference(pd.DatetimeIndex(train["date"].unique()))
print(f"Días del calendario sin registros: {len(dias_faltantes)}")
print(dias_faltantes.strftime("%m-%d").value_counts().to_string())
print("\nNulos en train:", int(train.isnull().sum().sum()), "| ventas negativas:", int((train["sales"] < 0).sum()))
assert (train["sales"] >= 0).all(), "Hay ventas negativas: log1p daría NaN y el RMSLE las ignoraría"
assert (test["date"].max() - fin_train).days <= HORIZONTE, "El test cubre más días que HORIZONTE: habría fechas sin predicción"

Reviso qué días faltan en el calendario, porque los rezagos que voy a construir dependen de un calendario completo. Si faltan días (por ejemplo, un día del año en que las tiendas cierran), los dejo como nulos en lugar de inventar ventas.

## 2. Exploración

In [ ]:
diario = train.groupby("date")["sales"].sum()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(diario.index, diario.values, lw=0.6, color="#93c5fd", label="Ventas diarias")
ax.plot(diario.index, diario.rolling(28, min_periods=1).mean(), lw=2, color="#1d4ed8", label="Media móvil 28 días")
terremoto = pd.Timestamp("2016-04-16")
if diario.index.min() <= terremoto <= diario.index.max():
    ax.axvline(terremoto, color="red", ls="--", lw=1, label="Terremoto (16-abr-2016)")
ax.set(title="Ventas totales diarias", xlabel="", ylabel="Ventas (unidades)")
ax.legend()
plt.tight_layout()
fig.savefig(IMG_DIR / "ventas_totales.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
por_dow = diario.groupby(diario.index.dayofweek).mean()
sns.barplot(x=[dias[i] for i in por_dow.index], y=por_dow.values, color="#3b82f6", ax=axes[0])
axes[0].set(title="Ventas diarias promedio por día de la semana", xlabel="", ylabel="Ventas (unidades)")

por_dia_mes = diario.groupby(diario.index.day).mean()
axes[1].plot(por_dia_mes.index, por_dia_mes.values, marker="o", color="#3b82f6")
axes[1].set(title="Ventas diarias promedio por día del mes", xlabel="Día del mes", ylabel="")
plt.tight_layout()
fig.savefig(IMG_DIR / "estacionalidad.png", dpi=130, bbox_inches="tight")
plt.show()

En esta competencia hay dos efectos de calendario que conviene vigilar: la **estacionalidad semanal** y el **día de pago** (el sector público paga el 15 y el último día del mes, según la descripción de los datos). Ambos los incluyo como variables del modelo.

In [ ]:
por_familia = train.groupby("family")["sales"].sum().sort_values(ascending=False)
top = por_familia.head(12)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top.values / por_familia.sum() * 100, y=top.index, color="#3b82f6", ax=ax)
ax.set(title="Participación en las ventas totales (12 familias principales)", xlabel="% de las ventas", ylabel="")
plt.tight_layout()
fig.savefig(IMG_DIR / "ventas_por_familia.png", dpi=130, bbox_inches="tight")
plt.show()

ultimos = train[train["date"] > fin_train - pd.Timedelta(days=VENTANA_CEROS)]
pct_ceros = ultimos.groupby(["store_nbr", "family"])["sales"].sum().eq(0).mean()
print(f"Series (tienda x familia) con ventas en cero durante los últimos {VENTANA_CEROS} días: {pct_ceros:.1%}")

promo = train.assign(con_promocion=train["onpromotion"] > 0).groupby("con_promocion")["sales"].agg(["mean", "size"])
display(promo.rename(columns={"mean": "ventas_promedio", "size": "registros"}))

La tabla de promociones es una **asociación**, no un efecto causal: las promociones se concentran en ciertas familias y tiendas, que ya venden distinto por sí solas.

## 3. Variables para el modelo

Predigo directamente los 16 días siguientes, así que cada variable tiene que estar disponible el día en que se hace la predicción:
- **Rezagos y medias móviles de las ventas:** solo uso valores con al menos `HORIZONTE` días de antigüedad. Así, para predecir cualquier día del horizonte, el dato más reciente que uso ya existe en la fecha de corte.
- **Calendario:** día de la semana, día del mes, mes, día de pago y feriados nacionales (excluyo los feriados trasladados, que se comportan como días normales, y marco aparte los eventos).
- **Promociones:** `onpromotion` viene en el conjunto de prueba, así que es legítimo usarla.
- **Tienda y producto:** tipo, clúster, ciudad, estado y familia.
- **Petróleo:** el precio con el mismo rezago mínimo.

In [ ]:
# Matriz de ventas: fechas x series (tienda, familia), con el calendario completo y los días del horizonte futuro
calendario_ext = pd.date_range(train["date"].min(), fin_train + pd.Timedelta(days=HORIZONTE), freq="D")
ventas = (train.set_index(["date", "store_nbr", "family"])["sales"]
               .unstack(["store_nbr", "family"])
               .reindex(calendario_ext))
lv = np.log1p(ventas)                    # trabajo en log1p(ventas) para optimizar RMSLE
T, S = lv.shape
print(f"Matriz de ventas: {T} días x {S} series")

# Rezagos y medias móviles (todos con antigüedad >= LAG_MIN)
variables = {}
for k in (LAG_MIN, 21, 28, 35):
    variables[f"lag_{k}"] = lv.shift(k)
base = lv.shift(LAG_MIN)
for w in (7, 14, 28, 56):
    variables[f"media_{w}d"] = base.rolling(w, min_periods=1).mean()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    mismo_dia = np.nanmean(np.stack([lv.shift(k).values for k in (21, 28, 35, 42)]), axis=0)
variables["media_mismo_dia_semana"] = pd.DataFrame(mismo_dia, index=lv.index, columns=lv.columns)

# Formato largo: una fila por (fecha, tienda, familia)
cols = lv.columns
largo = pd.DataFrame({
    "date": np.repeat(lv.index.values, S),
    "store_nbr": np.tile(cols.get_level_values(0).values, T),
    "family": np.tile(cols.get_level_values(1).values, T),
})
largo["y"] = lv.values.ravel().astype("float32")
for nombre, mat in variables.items():
    largo[nombre] = mat.values.ravel().astype("float32")

# Promociones (disponibles también en test)
promos = pd.concat([train[["date", "store_nbr", "family", "onpromotion"]],
                    test[["date", "store_nbr", "family", "onpromotion"]]])
largo = largo.merge(promos, on=["date", "store_nbr", "family"], how="left")

# Tienda
meta = stores.rename(columns={"type": "tipo_tienda"})
largo = largo.merge(meta, on="store_nbr", how="left")

# Calendario
f = largo["date"]
largo["dow"] = f.dt.dayofweek
largo["dia_mes"] = f.dt.day
largo["mes"] = f.dt.month
largo["semana_anio"] = f.dt.isocalendar().week.astype("int16")
largo["es_pago"] = ((f.dt.day == 15) | f.dt.is_month_end).astype("int8")

# Feriados nacionales (sin los trasladados) y eventos
h = holidays.copy()
h["transferred"] = h["transferred"].astype(str).str.lower().eq("true")
nacional = h[(h["locale"] == "National") & (~h["transferred"]) & h["type"].isin(["Holiday", "Transfer", "Additional", "Bridge"])]
largo["feriado_nacional"] = f.isin(nacional["date"]).astype("int8")
largo["evento"] = f.isin(h.loc[h["type"] == "Event", "date"]).astype("int8")

# Petróleo con rezago
petroleo = oil.set_index("date")["dcoilwtico"].reindex(calendario_ext).ffill().bfill()
largo["petroleo_lag"] = largo["date"].map(petroleo.shift(LAG_MIN)).astype("float32")

# Categóricas codificadas como enteros
cat_cols = ["store_nbr", "family", "tipo_tienda", "cluster", "city", "state"]
for c in cat_cols:
    largo[c + "_cod"] = largo[c].astype("category").cat.codes.astype("int16")

FEATURES = (list(variables) + ["onpromotion", "dow", "dia_mes", "mes", "semana_anio", "es_pago",
                               "feriado_nacional", "evento", "petroleo_lag"] + [c + "_cod" for c in cat_cols])
ES_CATEGORICA = [c.endswith("_cod") for c in FEATURES]
print(f"{len(FEATURES)} variables | {len(largo):,} filas")

## 4. Validación

Imito el problema real: la fecha de corte es `HORIZONTE` días antes del último día de entrenamiento. Entreno con lo anterior al corte y evalúo en esos últimos 16 días, con la misma métrica de la competencia. Una validación aleatoria sería engañosa porque mezclaría el futuro con el pasado.

In [ ]:
corte = fin_train - pd.Timedelta(days=HORIZONTE)
inicio = pd.Timestamp(FECHA_INICIO_ENTRENAMIENTO)

m_tr = (largo["date"] >= inicio) & (largo["date"] <= corte) & largo["y"].notna()
m_va = (largo["date"] > corte) & (largo["date"] <= fin_train) & largo["y"].notna()
print(f"Entrenamiento: {m_tr.sum():,} filas ({inicio:%Y-%m-%d} a {corte:%Y-%m-%d})")
print(f"Validación:    {m_va.sum():,} filas ({(corte + pd.Timedelta(days=1)):%Y-%m-%d} a {fin_train:%Y-%m-%d})")

def rmsle(y_log, pred_log):
    pred_log = np.clip(pred_log, 0, None)
    return float(np.sqrt(np.nanmean((pred_log - y_log) ** 2)))

y_va = largo.loc[m_va, "y"].to_numpy()

### Modelos de referencia

Antes del modelo de aprendizaje automático necesito saber cuánto se logra con reglas simples; si el modelo no las supera, no aporta.

In [ ]:
fechas_val = lv.loc[corte + pd.Timedelta(days=1): fin_train].index
ventana = lv.loc[corte - pd.Timedelta(days=27): corte]

# Referencia 1: media de los últimos 28 días (plana)
pred_plana = pd.DataFrame(np.tile(ventana.mean().values, (len(fechas_val), 1)), index=fechas_val, columns=lv.columns)

# Referencia 2: media de las últimas 4 semanas para el mismo día de la semana
por_dow = ventana.groupby(ventana.index.dayofweek).mean()
pred_dow = por_dow.reindex(fechas_val.dayofweek).set_axis(fechas_val, axis=0)

def pred_a_largo(pred_wide):
    """Lleva una matriz fechas x series al orden de las filas de validación."""
    pl = pd.DataFrame({
        "date": np.repeat(pred_wide.index.values, pred_wide.shape[1]),
        "store_nbr": np.tile(pred_wide.columns.get_level_values(0).values, pred_wide.shape[0]),
        "family": np.tile(pred_wide.columns.get_level_values(1).values, pred_wide.shape[0]),
        "pred": pred_wide.values.ravel(),
    })
    claves = largo.loc[m_va, ["date", "store_nbr", "family"]]
    return claves.merge(pl, on=["date", "store_nbr", "family"], how="left")["pred"].fillna(0).to_numpy()

p_plana = pred_a_largo(pred_plana)
p_dow = pred_a_largo(pred_dow)
print(f"RMSLE referencia 1 (media 28 días):        {rmsle(y_va, p_plana):.4f}")
print(f"RMSLE referencia 2 (mismo día de semana):  {rmsle(y_va, p_dow):.4f}")

### Gradient boosting

In [ ]:
def nuevo_modelo():
    return HistGradientBoostingRegressor(
        max_iter=MAX_ITER, learning_rate=LEARNING_RATE, max_leaf_nodes=MAX_LEAF_NODES,
        min_samples_leaf=MIN_SAMPLES_LEAF, l2_regularization=L2_REG, categorical_features=ES_CATEGORICA,
        early_stopping=False, random_state=SEMILLA)

X_tr, y_tr = largo.loc[m_tr, FEATURES], largo.loc[m_tr, "y"]
X_va = largo.loc[m_va, FEATURES]

modelo = nuevo_modelo().fit(X_tr, y_tr)
p_gb = np.clip(modelo.predict(X_va), 0, None)
print(f"RMSLE gradient boosting: {rmsle(y_va, p_gb):.4f}")

# Regla de negocio: si una serie no vendió nada en los últimos VENTANA_CEROS días, predigo 0
ventana_ceros = lv.loc[corte - pd.Timedelta(days=VENTANA_CEROS - 1): corte]
sin_ventas = ventana_ceros.fillna(0).sum().eq(0)
claves_va = largo.loc[m_va, ["store_nbr", "family"]]
es_cero = sin_ventas.reindex(pd.MultiIndex.from_frame(claves_va)).fillna(False).to_numpy()
p_gb_regla = np.where(es_cero, 0.0, p_gb)
print(f"RMSLE gradient boosting + regla de series en cero: {rmsle(y_va, p_gb_regla):.4f}")
print(f"Series afectadas por la regla: {sin_ventas.mean():.1%}")

## 5. Resultados

In [ ]:
resultados = pd.DataFrame({
    "modelo": ["Media 28 días", "Mismo día de la semana (4 semanas)", "Gradient boosting", "Gradient boosting + regla de ceros"],
    "RMSLE": [rmsle(y_va, p_plana), rmsle(y_va, p_dow), rmsle(y_va, p_gb), rmsle(y_va, p_gb_regla)],
}).sort_values("RMSLE").reset_index(drop=True)
resultados["mejora_vs_media_28d_%"] = (1 - resultados["RMSLE"] / rmsle(y_va, p_plana)) * 100
display(resultados)
resultados.to_csv("metricas.csv", index=False)

# La regla de ceros solo se usa en la submission si mejora el RMSLE en validación
USAR_REGLA_CEROS = rmsle(y_va, p_gb_regla) < rmsle(y_va, p_gb)
print(f"¿Uso la regla de ceros en la submission? {'sí' if USAR_REGLA_CEROS else 'no'}")

mejor = resultados.iloc[0]
print(f"Mejor modelo: {mejor['modelo']} con RMSLE {mejor['RMSLE']:.4f} "
      f"({mejor['mejora_vs_media_28d_%']:.1f} % menos error que la media de 28 días).")

In [ ]:
val = largo.loc[m_va, ["date", "store_nbr", "family"]].assign(real=y_va, pred=p_gb_regla if USAR_REGLA_CEROS else p_gb)

# Predicción vs. realidad en ventas totales (unidades)
real_diario = np.expm1(val["real"]).groupby(val["date"]).sum()
pred_diario = np.expm1(val["pred"]).groupby(val["date"]).sum()
contexto = diario.loc[corte - pd.Timedelta(days=45): corte]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(contexto.index, contexto.values, color="#93c5fd", label="Ventas reales (antes del corte)")
ax.plot(real_diario.index, real_diario.values, color="#1d4ed8", marker="o", label="Ventas reales (validación)")
ax.plot(pred_diario.index, pred_diario.values, color="#f59e0b", marker="o", label="Predicción")
ax.axvline(corte, color="gray", ls="--", lw=1)
ax.set(title="Predicción vs. realidad en ventas totales diarias", ylabel="Ventas (unidades)")
ax.legend()
plt.tight_layout()
fig.savefig(IMG_DIR / "prediccion_vs_real.png", dpi=130, bbox_inches="tight")
plt.show()

# Error por familia
err_familia = (val.assign(e2=(np.clip(val["pred"], 0, None) - val["real"]) ** 2)
                  .groupby("family")["e2"].mean().pow(0.5).sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(x=err_familia.values, y=err_familia.index, color="#3b82f6", ax=ax)
ax.set(title="RMSLE por familia de producto (validación)", xlabel="RMSLE", ylabel="")
plt.tight_layout()
fig.savefig(IMG_DIR / "error_por_familia.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
# Importancia de variables por permutación (sobre una muestra de validación)
rng = np.random.default_rng(SEMILLA)
idx = rng.choice(len(X_va), size=min(20000, len(X_va)), replace=False)
imp = permutation_importance(modelo, X_va.iloc[idx], y_va[idx], n_repeats=3, random_state=SEMILLA,
                             scoring="neg_root_mean_squared_error")
imp = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=imp.values, y=imp.index, color="#3b82f6", ax=ax)
ax.set(title="Importancia por permutación (aumento del RMSLE al desordenar la variable)", xlabel="Aumento del RMSLE", ylabel="")
plt.tight_layout()
fig.savefig(IMG_DIR / "importancia_variables.png", dpi=130, bbox_inches="tight")
plt.show()

Cuando el error de una familia sea alto, lo primero que reviso es si sus series tienen muchos ceros o cambios bruscos de nivel. Una familia con ventas casi nulas puede dar un RMSLE alto aunque venda poco, y no siempre es un fallo del modelo.

## 6. Predicciones para Kaggle

Reentreno con todos los datos disponibles y predigo los 16 días del archivo `test.csv`. El resultado queda en `submission.csv` (no se sube a este repositorio); se envía a la competencia para obtener el puntaje en la tabla pública.

In [ ]:
if GENERAR_SUBMISSION:
    m_all = (largo["date"] >= inicio) & (largo["date"] <= fin_train) & largo["y"].notna()
    m_te = largo["date"] > fin_train

    modelo_final = nuevo_modelo().fit(largo.loc[m_all, FEATURES], largo.loc[m_all, "y"])
    p_te = np.clip(modelo_final.predict(largo.loc[m_te, FEATURES]), 0, None)

    claves_te = largo.loc[m_te, ["date", "store_nbr", "family"]]
    if USAR_REGLA_CEROS:
        # misma regla de ceros, con los últimos VENTANA_CEROS días de train
        vent_final = lv.loc[fin_train - pd.Timedelta(days=VENTANA_CEROS - 1): fin_train]
        cero_final = vent_final.fillna(0).sum().eq(0)
        es_cero_te = cero_final.reindex(pd.MultiIndex.from_frame(claves_te[["store_nbr", "family"]])).fillna(False).to_numpy()
        p_te = np.where(es_cero_te, 0.0, p_te)

    pred_te = claves_te.assign(sales=np.expm1(p_te))
    sub = test[["id", "date", "store_nbr", "family"]].merge(pred_te, on=["date", "store_nbr", "family"], how="left")
    assert sub["sales"].notna().all(), "Hay filas del test sin predicción"
    sub["sales"] = sub["sales"].clip(lower=0)
    sub[["id", "sales"]].to_csv("submission.csv", index=False)
    print(f"submission.csv: {len(sub):,} filas | ventas predichas totales: {sub['sales'].sum():,.0f}")